# 01 — Giriş Problemi: Bulut Sunucusu

**Öğrenme hedefleri:**
1. Decision variable, objective function ve constraint kavramlarını somut bir problem üzerinden anlamak
2. Feasible region'ı grafikle görmek
3. LP (sürekli) ile ILP (tam sayı) arasındaki farkı gözlemlemek
4. Infeasible ve unbounded durumları tanımak

**Tahmini süre:** ~30 dakika

In [ ]:
import sys
if "google.colab" in sys.modules:
    !pip install -q pulp ipywidgets
    !git clone -q https://github.com/Meliksahcaliskan/ILP_Workshop.git
    sys.path.insert(0, "ILP_Workshop")

In [ ]:
import pulp
import numpy as np
import matplotlib.pyplot as plt
from utils.plotting import plot_feasible_region_cloud, show_status

---
## Problem Tanımı

Bir şirket iki tip bulut sunucusu kiralayabilir.  
Toplamda en az **30 vCPU** ve **90 GB RAM** gerekiyor.  
**Hedef:** Saatlik maliyeti en aza indirmek için her tipten kaç sunucu kiralanmalı?

| Tip | vCPU | RAM (GB) | Saatlik maliyet (\$) |
|-----|------|----------|----------------------|
| A   | 4    | 8        | 10                   |
| B   | 8    | 32       | 25                   |

---
## ILP'nin Yapı Taşları

| Kavram | Bu problemdeki karşılığı |
|--------|-------------------------|
| **Decision Variable** | `a` = Tip-A sayısı, `b` = Tip-B sayısı |
| **Objective Function** | min `10a + 25b` (saatlik maliyet) |
| **Constraint** | `4a + 8b ≥ 30` (vCPU), `8a + 32b ≥ 90` (RAM) |

---
## Adım 1 — LP Çözümü (sürekli değişkenler)

In [ ]:
prob_lp = pulp.LpProblem("cloud_LP", pulp.LpMinimize)
a = pulp.LpVariable("a", lowBound=0, cat="Continuous")
b = pulp.LpVariable("b", lowBound=0, cat="Continuous")

prob_lp += 10*a + 25*b
prob_lp += 4*a + 8*b >= 30, "vcpu"
prob_lp += 8*a + 32*b >= 90, "ram"

prob_lp.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_lp)

a_lp, b_lp = pulp.value(a), pulp.value(b)
print(f"a = {a_lp:.4f},  b = {b_lp:.4f}")
print(f"Maliyet = ${10*a_lp + 25*b_lp:.4f}")

> **Soru:** 3.75 sunucu kiralayabilir miyiz? Bu sonuç pratikte ne anlama geliyor?

---
## Adım 2 — Feasible Region Grafiği

In [ ]:
fig = plot_feasible_region_cloud(a_lp=a_lp, b_lp=b_lp)
plt.show()

---
## Adım 2b — İnteraktif Keşif (ipywidgets)

> Sliderları hareket ettirerek: **A ve B maliyetlerini** ve **ihtiyaç değerlerini** değiştir.  
> Objective eğimi değiştikçe optimum nokta nerede kalıyor?

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display as ipy_display

    def interactive_cloud(cost_a, cost_b, vcpu_need, ram_need):
        p = pulp.LpProblem("cloud_interactive", pulp.LpMinimize)
        ai = pulp.LpVariable("a", lowBound=0, cat="Continuous")
        bi = pulp.LpVariable("b", lowBound=0, cat="Continuous")
        p += cost_a*ai + cost_b*bi
        p += 4*ai + 8*bi >= vcpu_need
        p += 8*ai + 32*bi >= ram_need
        p.solve(pulp.PULP_CBC_CMD(msg=0))

        a_val = pulp.value(ai)
        b_val = pulp.value(bi)
        status = pulp.LpStatus[p.status]

        import numpy as np
        import matplotlib.pyplot as plt
        from utils.plotting import PALETTE

        fig, ax = plt.subplots(figsize=(7, 6))
        a_vals = np.linspace(0, 20, 400)

        b_cpu = (vcpu_need - 4*a_vals) / 8
        b_ram = (ram_need  - 8*a_vals) / 32

        ax.plot(a_vals, b_cpu, color=PALETTE["blue"], lw=2, label="vCPU kısıtı")
        ax.plot(a_vals, b_ram, color=PALETTE["orange"], lw=2, label="RAM kısıtı")

        b_feas = np.maximum(np.maximum(b_cpu, b_ram), 0)
        ax.fill_between(a_vals, b_feas, 15, alpha=0.12, color=PALETTE["green"])

        if status == "Optimal" and a_val is not None:
            # Objective level curves
            for lvl in [pulp.value(p.objective) * m for m in [1.0, 1.5, 2.0]]:
                b_obj = (lvl - cost_a*a_vals) / cost_b
                ax.plot(a_vals, b_obj, "--", color=PALETTE["gray"], lw=0.8, alpha=0.6)
            ax.scatter([a_val], [b_val], s=160, color=PALETTE["vermil"], zorder=5,
                       label=f"LP opt ({a_val:.2f}, {b_val:.2f})\n${pulp.value(p.objective):.2f}")

        ax.set_xlim(0, 15); ax.set_ylim(0, 12)
        ax.set_xlabel("a (Tip-A sayısı)"); ax.set_ylabel("b (Tip-B sayısı)")
        ax.set_title(f"Durum: {status}  |  Maliyet: A=${cost_a}, B=${cost_b}")
        ax.legend(fontsize=8); ax.grid(True, linestyle="--", alpha=0.3)
        plt.tight_layout(); plt.show()

    w = widgets.interactive(
        interactive_cloud,
        cost_a=widgets.IntSlider(value=10, min=5, max=40, step=1,
                                  description="Maliyet A ($)"),
        cost_b=widgets.IntSlider(value=25, min=5, max=60, step=5,
                                  description="Maliyet B ($)"),
        vcpu_need=widgets.IntSlider(value=30, min=10, max=60, step=5,
                                     description="vCPU ihtiyacı"),
        ram_need=widgets.IntSlider(value=90, min=30, max=150, step=10,
                                    description="RAM ihtiyacı"),
    )
    ipy_display(w)
except ImportError:
    print("ipywidgets yüklü değil. Aşağıdaki statik hücreyi kullanın.")

In [ ]:
# Yedek: ipywidgets çalışmıyorsa bu hücrede değerleri manuel değiştir
COST_A, COST_B = 10, 25
VCPU_NEED, RAM_NEED = 30, 90

fig = plot_feasible_region_cloud(a_lp=a_lp, b_lp=b_lp)
plt.title(f"Maliyet A={COST_A}, B={COST_B} | vCPU≥{VCPU_NEED}, RAM≥{RAM_NEED}")
plt.show()

---
## Adım 3 — ILP Çözümü (tam sayı kısıtı)

In [ ]:
prob_ilp = pulp.LpProblem("cloud_ILP", pulp.LpMinimize)
a_i = pulp.LpVariable("a", lowBound=0, cat="Integer")
b_i = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_ilp += 10*a_i + 25*b_i
prob_ilp += 4*a_i + 8*b_i >= 30
prob_ilp += 8*a_i + 32*b_i >= 90

prob_ilp.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_ilp)

a_ilp, b_ilp = int(pulp.value(a_i)), int(pulp.value(b_i))
print(f"a = {a_ilp},  b = {b_ilp}")
print(f"Maliyet = ${10*a_ilp + 25*b_ilp:.2f}")

---
## Adım 4 — LP vs ILP Karşılaştırması

In [ ]:
fig = plot_feasible_region_cloud(a_lp=a_lp, b_lp=b_lp, a_ilp=a_ilp, b_ilp=b_ilp)
plt.show()

print(f"LP  maliyet: ${10*a_lp + 25*b_lp:.3f}  (alt sınır)")
print(f"ILP maliyet: ${10*a_ilp + 25*b_ilp:.2f}")
print(f"Fark       : ${10*a_ilp + 25*b_ilp - (10*a_lp + 25*b_lp):.3f}")

---
## Adım 5 — Aşağı Yuvarlama Neden Çalışmaz?

In [ ]:
a_round, b_round = 3, 1

vcpu_ok = (4*a_round + 8*b_round >= 30)
ram_ok  = (8*a_round + 32*b_round >= 90)

print(f"a=3, b=1 için:")
print(f"  vCPU: {4*a_round + 8*b_round} >= 30 ? {'✅' if vcpu_ok else '❌'}")
print(f"  RAM : {8*a_round + 32*b_round} >= 90 ? {'✅' if ram_ok else '❌'}")
print()
print("Kırmızı X ile grafikte gösterelim:")

fig = plot_feasible_region_cloud(
    a_lp=a_lp, b_lp=b_lp, a_ilp=a_ilp, b_ilp=b_ilp,
    infeasible_points=[(a_round, b_round)]
)
plt.show()
print("=> Aşağı yuvarlama infeasible bir nokta üretebilir — solver bu yüzden gerekli!")

---
## 🔧 Değiştir-Gözlemle Görevleri

### Görev 1 — Ek Kısıt: En Fazla 1 Tip-B Sunucu

> **Tahmin:** `b ≤ 1` kısıtını eklersen maliyet nasıl değişir?

In [ ]:
prob_g1 = pulp.LpProblem("cloud_g1", pulp.LpMinimize)
ag1 = pulp.LpVariable("a", lowBound=0, cat="Integer")
bg1 = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_g1 += 10*ag1 + 25*bg1
prob_g1 += 4*ag1 + 8*bg1 >= 30
prob_g1 += 8*ag1 + 32*bg1 >= 90
prob_g1 += bg1 <= 1  # YENİ KISIT

prob_g1.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g1)
print(f"a={int(pulp.value(ag1))}, b={int(pulp.value(bg1))}, maliyet=${pulp.value(prob_g1.objective):.2f}")

> **Eğitmen notu:** Beklenen `a=8, b=1`, maliyet `$105`. B sayısı kısıtlanınca RAM ihtiyacı için A sunucuları devreye giriyor.

### Görev 2 — Bütçe Kısıtı: Saatlik En Fazla \$80

> **Tahmin:** LP minimumu \$84.375 iken \$80 bütçesiyle ne olur?

In [ ]:
prob_g2 = pulp.LpProblem("cloud_g2", pulp.LpMinimize)
ag2 = pulp.LpVariable("a", lowBound=0, cat="Integer")
bg2 = pulp.LpVariable("b", lowBound=0, cat="Integer")

prob_g2 += 10*ag2 + 25*bg2
prob_g2 += 4*ag2 + 8*bg2 >= 30
prob_g2 += 8*ag2 + 32*bg2 >= 90
prob_g2 += 10*ag2 + 25*bg2 <= 80  # BÜTÇE KISITI

prob_g2.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g2)

> **Eğitmen notu:** **Infeasible** — LP minimumu bile \$84.375 olduğundan \$80 bütçesiyle ihtiyaçları karşılamak mümkün değil.

### Görev 3 — LpMaximize: Ne Olur?

> **Tahmin:** Minimize yerine maximize yaparsak?

In [ ]:
prob_g3 = pulp.LpProblem("cloud_g3", pulp.LpMaximize)
ag3 = pulp.LpVariable("a", lowBound=0)
bg3 = pulp.LpVariable("b", lowBound=0)

prob_g3 += 10*ag3 + 25*bg3
prob_g3 += 4*ag3 + 8*bg3 >= 30
prob_g3 += 8*ag3 + 32*bg3 >= 90

prob_g3.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g3)

> **Eğitmen notu:** **Unbounded** — '≥' kısıtlarıyla maximize yaparken `a` ve `b` sonsuz artabilir. Modelleme hatası göstergesi.

### Görev 4 — Üçüncü Sunucu Tipi (Serbest)

> Kendi seçtiğin değerlerle Tip-C sunucu ekle.  
> Grafik artık 2D çizilemez — solver'ın gücü burada ortaya çıkıyor!

In [ ]:
# Tip-C değerlerini doldur
vcpu_c = 6   # değiştir
ram_c  = 16  # değiştir
cost_c = 15  # değiştir

prob_g4 = pulp.LpProblem("cloud_g4", pulp.LpMinimize)
a4 = pulp.LpVariable("a", lowBound=0, cat="Integer")
b4 = pulp.LpVariable("b", lowBound=0, cat="Integer")
c4 = pulp.LpVariable("c", lowBound=0, cat="Integer")

prob_g4 += 10*a4 + 25*b4 + cost_c*c4
prob_g4 += 4*a4 + 8*b4 + vcpu_c*c4 >= 30
prob_g4 += 8*a4 + 32*b4 + ram_c*c4  >= 90

prob_g4.solve(pulp.PULP_CBC_CMD(msg=0))
show_status(prob_g4)
print(f"a={int(pulp.value(a4))}, b={int(pulp.value(b4))}, c={int(pulp.value(c4))}")
print(f"Maliyet: ${pulp.value(prob_g4.objective):.2f}")